# Fine-tune PE on RSTPReid + TVPReid train (Kaggle)

**Run A (default):** full fine-tune PE-B/16.  
**Run B:** PE-L/14 last-4 blocks (or LoRA).

This notebook **downloads** the data (Internet on). No need to attach RSTPReid/TVPReid as Kaggle datasets.

- Uses official **train + val** only (ID-disjoint). **Never** Alexandria / GroOT / test splits.
- After training, download `best.pt` and eval locally with `scripts/eval_finetuned.py`.
- TVPReid metrics become **in-domain** if you later score TVPReid test with this checkpoint; Alexandria/GroOT stay out-of-domain.

In [ ]:
from pathlib import Path

RUN = "A"  # "A" = PE-B full FT; "B" = PE-L last-4
USE_LORA = False
EPOCHS = 10 if RUN == "A" else 8
MAX_STEPS = None  # e.g. 50 for a smoke run
FRAMES_PER_VIDEO = 4  # TVPReid stills per clip
BATCH = 64 if RUN == "A" else 16

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("results/finetune")
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "finetune_data"
print("run", RUN, "epochs", EPOCHS, "work", WORK)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"

def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    if shawaf_dir.exists():
        run(["git", "fetch", "origin"], cwd=shawaf_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=shawaf_dir)
    else:
        run(["git", "clone", SHAWAF_URL, str(shawaf_dir)])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]", "gdown"])
    ROOT = shawaf_dir
else:
    ROOT = Path("..").resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
print("ROOT", ROOT)

## Download + package (RSTPReid, TVPReid train/val)

Enable **Internet** in the Kaggle notebook settings. First run can take a while (Drive + Hub videos).

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "prepare_finetune_kaggle.py"),
    "--out", str(DATA),
    "--frames-per-video", str(FRAMES_PER_VIDEO),
]
print("+", " ".join(cmd), flush=True)
subprocess.check_call(cmd)

train_files = sorted(DATA.glob("*/train.jsonl"))
val_files = sorted(DATA.glob("*/val.jsonl"))
print("train", train_files)
print("val", val_files)
assert any("rstpreid" in str(p) for p in train_files), "RSTPReid train missing"
assert any("tvpreid" in str(p) for p in train_files), "TVPReid train missing"

## Train

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from shawaf_vlm.finetune.train import FinetuneConfig, train_pe

if RUN == "A":
    cfg = FinetuneConfig(
        base_key="pe_core_b16",
        output_dir=str(WORK / "pe_b16_run_a"),
        epochs=EPOCHS,
        batch_size=BATCH,
        lr=5e-6,
        image_size=224,
        fit="letterbox",
        device="cuda",
    )
else:
    cfg = FinetuneConfig(
        base_key="pe_core_l14",
        output_dir=str(WORK / "pe_l14_run_b"),
        epochs=EPOCHS,
        batch_size=BATCH,
        lr=2e-6,
        image_size=336,
        fit="letterbox",
        unfreeze_last_blocks=None if USE_LORA else 4,
        use_lora=USE_LORA,
        grad_checkpoint=True,
        device="cuda",
    )

cfg.train_jsonl = [str(p) for p in train_files]
cfg.val_jsonl = [str(p) for p in val_files]
cfg.image_root = str(DATA)
cfg.max_steps = MAX_STEPS
cfg.num_workers = 2

best = train_pe(cfg)
env = "SHAWAF_PE_B16_FT" if cfg.base_key == "pe_core_b16" else "SHAWAF_PE_L14_FT"
print("best", best)
print(f"Download best.pt then: set {env}=<path>; python scripts/eval_finetuned.py --model {cfg.base_key}_ft --fit letterbox --max-frames 16")